```markdown
# What Are Runnables in LangChain?

Before diving into *why* Runnables exist, let’s first understand **what they are** and **how they work**.

---

## What Is a Runnable?
A **Runnable** is the **core building block** in LangChain.  
Think of it as a **unit of work** that can be executed, streamed, batched, or composed with other units.  

- Every component (Prompt, LLM, Parser, Retriever, Tool) is a Runnable.  
- Runnables expose a **common interface**:  
  - `.invoke()` → run once with input.  
  - `.batch()` → run multiple inputs at once.  
  - `.stream()` → stream partial outputs.  

---

## Why LangChain Introduced Runnables
LangChain originally had **Chains** (like `LLMChain`, `SequentialChain`).  
But these were limited — each chain type had its own API, making composition harder.  

Runnables were introduced to:
- **Unify execution** → everything behaves the same way.  
- **Simplify composition** → connect components with `|` operator.  
- **Enable streaming** → partial outputs flow naturally.  
- **Increase flexibility** → mix prompts, LLMs, parsers, retrievers, and even Python functions.  

In short: **Chains were recipes. Runnables are Lego blocks.**  
You can snap them together in any order to build your own workflow.

---

## Where Do We Use Runnables?
- **Prompt → LLM → Parser** pipelines (e.g., `prompt | llm | JsonOutputParser()`).  
- **Parallel tasks** (e.g., run multiple queries at once).  
- **Custom functions** wrapped as Runnables (`RunnableLambda`).  
- **Streaming workflows** where outputs are consumed live.  
- **Serialization** of pipelines for saving and reloading.

---

## Types of Runnables

1. **RunnableSequence**  
   - Chains multiple Runnables together.  
   - Example: `prompt | llm | parser`.  
   - Most common type — replaces `LLMChain`.

2. **RunnableParallel**  
   - Runs multiple Runnables at the same time.  
   - Example: send one query to two LLMs and compare outputs.

3. **RunnableLambda**  
   - Wraps a Python function into a Runnable.  
   - Example: `lambda x: x.upper()` inside a pipeline.

4. **RunnableGenerator**  
   - Wraps a generator function for streaming.  
   - Example: stream tokens or JSON fragments as they’re produced.

5. **RunnableBinding**  
   - Adds extra functionality around an existing Runnable.  
   - Example: bind default parameters or logging.

6. **RunnableSerializable**  
   - Makes a Runnable exportable to JSON.  
   - Useful for saving and reloading pipelines.

7. **RunnableConfig**  
   - Configuration object for controlling runtime behavior.  
   - Example: set concurrency limits or tracing options.

---

## Chains vs Runnables

| Feature            | Chains (Old)         | Runnables (Now) |
|--------------------|----------------------|-----------------|
| Abstraction        | Different classes    | Unified API     |
| Composition        | Limited (LLMChain, SequentialChain) | Flexible (`|` operator) |
| Streaming          | Not native           | Native support  |
| Extensibility      | Hard to extend       | Easy to wrap any function |
| Status (2026)      | Deprecated           | Standard        |

---

## Takeaway
- **Chains are gone** — replaced by Runnables.  
- Runnables unify everything under one interface (`invoke`, `batch`, `stream`).  
- You use Runnables for prompts, LLMs, parsers, retrievers, and even custom Python functions.  
- This makes LangChain pipelines **more powerful, flexible, and production‑ready**.
```

# RunnableSequence (The Pipeline Backbone)

**What it does:**  
Chains multiple Runnables sequentially. The output of one step automatically becomes the input of the next.

**Why it's used in web apps:**  
It is the foundational construct for almost every LLM feature. The standard web app flow usually looks like:  
`PromptTemplate → ChatModel → OutputParser`.

**Web App Benefit:**  
It natively supports the `.stream()` method. This allows web apps to stream token‑by‑token text directly to the frontend (like ChatGPT) instead of waiting for the full response to generate.

---

## Another Example Use Case
Instead of translation, let’s build a **summarization pipeline**:  
- Step 1: Prompt formats the user’s text into a summarization request.  
- Step 2: LLM generates the summary.  
- Step 3: Parser extracts the plain string output.  

This shows how RunnableSequence can be applied to tasks beyond translation.

In [23]:
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
import os
from dotenv import load_dotenv

# Load API key
load_dotenv()
groq_key = os.getenv("GROQ_API_KEY")

# Step 1: Prompt for summarization
prompt = PromptTemplate.from_template("Summarize the following text in 2-3 sentences:\n\n{text}")

# Step 2: LLM
llm = ChatGroq(model="llama-3.1-8b-instant", api_key=groq_key)

# Step 3: Parser
parser = StrOutputParser()

# RunnableSequence pipeline
chain = prompt | llm | parser

# Run the pipeline
input_text = """
LangChain provides a unified interface for working with language models. 
It allows developers to build applications that combine prompts, models, parsers, and retrievers 
into flexible pipelines. By introducing Runnables, LangChain simplified composition and enabled streaming.
"""
result = chain.invoke({"text": input_text})
print("Summary:\n", result)

# Stream the pipeline (simulate frontend typing effect)
streamed_text = ""
for chunk in chain.stream({"text": input_text}):
    streamed_text += chunk
    print(streamed_text, end="\r", flush=True)

print("\n\nFinal Streamed Summary:\n", streamed_text)

Summary:
 Here's a summary of the text in 2-3 sentences:

LangChain offers a unified interface for working with language models, enabling developers to build complex applications. It combines various components such as prompts, models, parsers, and retrievers into flexible pipelines. LangChain's introduction of Runnables simplifies composition and allows for streaming capabilities.
Here's a summary of the text in 2-3 sentences:

Here's a summary of the text in 2-3 sentences:

Here's a summary of the text in 2-3 sentences:

Here's a summary of the text in 2-3 sentences:

Here's a summary of the text in 2-3 sentences:

Here's a summary of the text in 2-3 sentences:

Here's a summary of the text in 2-3 sentences:

Here's a summary of the text in 2-3 sentences:

Here's a summary of the text in 2-3 sentences:

Here's a summary of the text in 2-3 sentences:ng

Here's a summary of the text in 2-3 sentences:ng with

Here's a summary of the text in 2-3 sentences:ng with language

Here's a summa

In [25]:
# Streaming demo: print chunks as they arrive
print("Streaming Output:\n")
for chunk in chain.stream({"text": input_text}):
    # print each chunk immediately
    print(chunk, end="", flush=True)

print("\n\n--- End of Stream ---")

Streaming Output:

LangChain offers a unified interface for working with language models, enabling developers to build applications that combine various tools and components into flexible pipelines. This framework simplifies composition through the use of Runnables and also supports streaming capabilities. As a result, developers can create more complex and dynamic applications with ease.

--- End of Stream ---


# RunnablePassthrough (The Context Carrier)

**What it does:**  
Forwards the input data untouched or adds extra keys to it while passing it to the next step in the chain.

**Why it's used in web apps:**  
It’s vital for **Retrieval‑Augmented Generation (RAG)**. When a user asks a question, you often need to pass both the **original query** and the **retrieved background documents** into the prompt. RunnablePassthrough ensures that nothing gets dropped between steps.

**Web App Benefit:**  
Prevents data loss without writing messy custom wrapper functions. It acts like a “carrier” that keeps context intact while moving through the pipeline.

In [26]:
# RunnablePassthrough Example
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
import os
from dotenv import load_dotenv

# Load API key
load_dotenv()
groq_key = os.getenv("GROQ_API_KEY")

# Step 1: RunnablePassthrough (carry user question forward)
passthrough = RunnablePassthrough()

# Step 2: Prompt that expects both question and context
prompt = PromptTemplate.from_template(
    "Answer the question using the context.\n\nContext: {context}\n\nQuestion: {question}"
)

# Step 3: LLM
llm = ChatGroq(model="llama-3.1-8b-instant", api_key=groq_key)

# Step 4: Parser
parser = StrOutputParser()

# Build pipeline: passthrough → prompt → llm → parser
chain = {"question": passthrough, "context": passthrough} | prompt | llm | parser

# Run the pipeline
result = chain.invoke({"question": "What is LangChain?", "context": "LangChain is a framework for building LLM apps."})
print("Output:\n", result)


Output:
 LangChain is a framework for building LLM (Large Language Model) apps.


# RunnableParallel (The Performance Booster)

**What it does:**  
Executes multiple operations concurrently and returns the combined results as a dictionary.

**Why it's used in web apps:**  
Web applications often need to perform multiple independent tasks at once — for example, searching a vector database, fetching data from an SQL database, and checking user permissions. Running these sequentially would slow down the app.

**Web App Benefit:**  
By fanning out independent execution branches in parallel, it drastically reduces HTTP request response times and improves user experience.

In [27]:
# RunnableParallel Example
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel
import os
from dotenv import load_dotenv

# Load API key
load_dotenv()
groq_key = os.getenv("GROQ_API_KEY")

# Define two prompts
prompt_summary = PromptTemplate.from_template("Summarize this text in one sentence:\n\n{text}")
prompt_translate = PromptTemplate.from_template("Translate this text into French:\n\n{text}")

# LLM
llm = ChatGroq(model="llama-3.1-8b-instant", api_key=groq_key)

# Parser
parser = StrOutputParser()

# Pipelines
summary_chain = prompt_summary | llm | parser
translate_chain = prompt_translate | llm | parser

# RunnableParallel executes both at once
parallel_chain = RunnableParallel({
    "summary": summary_chain,
    "translation": translate_chain
})

# Run the parallel tasks
input_text = "LangChain introduced Runnables to unify execution and enable streaming."
result = parallel_chain.invoke({"text": input_text})

print("Parallel Output:\n")
print("Summary:", result["summary"])
print("Translation:", result["translation"])

Parallel Output:

Summary: LangChain introduced "Runnables" to unify the execution of tasks and enable real-time data streaming.
Translation: La traduction du texte en français est :

LangChain a introduit Runnables pour unifier l'exécution et permettre le streaming.


# RunnableLambda (The Custom Logic Bridge)

**What it does:**  

Wraps a standard Python function so that it adopts the Runnable interface.

**Why it's used in web apps:**  

LLMs don’t operate in isolation, web apps often need **custom business logic** between steps. For example, cleaning user input, logging activity, parsing session headers, or calling internal APIs. RunnableLambda lets you insert this logic directly into a LangChain pipeline without breaking the flow.

**Web App Benefit:**  

It allows developers to seamlessly integrate standard codebase logic into LCEL pipes (`|`), making pipelines flexible and production‑ready.

In [28]:
# RunnableLambda Example
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
import os
from dotenv import load_dotenv

# Load API key
load_dotenv()
groq_key = os.getenv("GROQ_API_KEY")

# Step 1: Custom preprocessing logic
def clean_input(x: dict) -> dict:
    # Example: strip whitespace and capitalize
    return {"sentence": x["sentence"].strip().capitalize()}

clean_runnable = RunnableLambda(clean_input)

# Step 2: Prompt
prompt = PromptTemplate.from_template("Translate this sentence into Spanish: {sentence}")

# Step 3: LLM
llm = ChatGroq(model="llama-3.1-8b-instant", api_key=groq_key)

# Step 4: Parser
parser = StrOutputParser()

# Build pipeline: custom logic → prompt → llm → parser
chain = clean_runnable | prompt | llm | parser

# Run the pipeline
result = chain.invoke({"sentence": "   hello world   "})
print("Output:\n", result)

Output:
 The translation of "Hello world" into Spanish is "Hola mundo".
